# Prediction When Features Outnumber Samples

**Team:** Byte Busters, Batch A1  
**Course:** Machine Learning Lab (AIP301), IIIT Kota  
**November 2026**

This notebook contains all experiments for the high-dimensional regression project.

## Setup and Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_loader import load_dataset, audit_summary
from data_audit import data_dictionary, basic_audit, plot_target_distribution, find_leakage_candidates
from preprocessing import train_val_split, repeated_splits, make_preprocessing_pipeline
from feature_engineering import feature_sets_by_ratio, add_irrelevant_features, add_redundant_features
from models import ALL_MODELS, ridge_regression, lasso_regression, random_forest_regressor
from evaluation import regression_metrics
from experiments import ExperimentLogger, nested_cv, INNER_GRID
from visualization import plot_learning_curve, plot_performance_vs_pn, plot_repeated_split

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

print("Imports complete")

## Phase 1-2: Load and Audit Data

In [ ]:
# load dataset
X, y, feature_names, target_name = load_dataset()

print(f"Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Target: {target_name}")
print(f"p/n ratio: {X.shape[1] / X.shape[0]:.3f}")

In [ ]:
# audit summary
audit = basic_audit(X, y, include_pairwise_corr=False)
print("Dataset Audit:")
for k, v in audit.items():
    print(f"  {k}: {v}")

In [ ]:
# plot target distribution
plot_target_distribution(y)
print("Target distribution plot saved")

In [ ]:
# check for potential leakage
leakage = find_leakage_candidates(X, y, threshold=0.9)
print(f"Features with |correlation| >= 0.9: {len(leakage)}")
if leakage:
    print("Top candidates:")
    for feat, corr in list(leakage.items())[:5]:
        print(f"  {feat}: {corr:.4f}")

## Phase 3: Train/Val Split

In [ ]:
X_train, X_val, y_train, y_val = train_val_split(X, y, test_size=0.2)
print(f"Train: {X_train.shape[0]} samples")
print(f"Val: {X_val.shape[0]} samples")

## Phase 4: Feature Sets by p/n Ratio

In [ ]:
feature_sets = feature_sets_by_ratio(X_train)
for name, fset in feature_sets.items():
    print(f"{name}: p={fset['p']}, n={fset['n']}, p/n={fset['p_over_n']:.3f}")

## Phase 5: Model Comparison

In [ ]:
logger = ExperimentLogger()

print("Running model comparison...")
model_results = {}

for model_name, model_func in ALL_MODELS.items():
    if model_name == "Linear Regression":  # skip unstable model
        continue
    
    print(f"\nTesting {model_name}...")
    
    # simple single split test
    pipeline = Pipeline([
        ('scaler', make_preprocessing_pipeline().steps[0][1]),
        ('model', model_func())
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    metrics = regression_metrics(y_val, y_pred)
    
    print(f"  R²: {metrics['r2']:.4f}")
    print(f"  RMSE: {metrics['rmse']:.2f}")
    print(f"  MAE: {metrics['mae']:.2f}")
    
    logger.log(
        experiment_id="model_comparison",
        experiment_type="single_split",
        model=model_name,
        metrics=metrics,
        training_sample_count=X_train.shape[0],
        original_feature_count=X_train.shape[1]
    )

print("\nModel comparison complete")

## Phase 6: Repeated Splits

In [ ]:
print("Running repeated splits evaluation...")
n_splits = 10
repeated_results = {}

for model_name in ["Ridge", "Lasso", "Random Forest"]:
    model_func = ALL_MODELS[model_name]
    scores = []
    
    for i, (Xtr, Xv, ytr, yv) in enumerate(repeated_splits(X, y, n_splits=n_splits)):
        pipeline = Pipeline([
            ('scaler', make_preprocessing_pipeline().steps[0][1]),
            ('model', model_func())
        ])
        pipeline.fit(Xtr, ytr)
        pred = pipeline.predict(Xv)
        metrics = regression_metrics(yv, pred)
        scores.append(metrics['r2'])
    
    repeated_results[model_name] = scores
    print(f"{model_name}: mean R² = {np.mean(scores):.4f} ± {np.std(scores):.4f}")

print("\nRepeated splits complete")

## Phase 7: Nested Cross-Validation

In [ ]:
print("Running nested CV for Ridge...")
nested_results = nested_cv(
    X, y,
    model_func=ridge_regression,
    param_grid=INNER_GRID['Ridge'],
    inner_cv=3,
    outer_cv=5
)

print(f"Mean R²: {nested_results['mean_r2']:.4f}")
print(f"Std R²: {nested_results['std_r2']:.4f}")
print(f"Outer scores: {nested_results['outer_scores']}")

## Phase 8: Learning Curve

In [ ]:
print("Running learning curve experiment...")
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
lc_results = []

for frac in fractions:
    n_samples = int(len(X_train) * frac)
    Xtr_subset = X_train.iloc[:n_samples]
    ytr_subset = y_train.iloc[:n_samples]
    
    pipeline = Pipeline([
        ('scaler', make_preprocessing_pipeline().steps[0][1]),
        ('model', ridge_regression())
    ])
    pipeline.fit(Xtr_subset, ytr_subset)
    
    train_pred = pipeline.predict(Xtr_subset)
    val_pred = pipeline.predict(X_val)
    
    train_r2 = regression_metrics(ytr_subset, train_pred)['r2']
    val_r2 = regression_metrics(y_val, val_pred)['r2']
    
    lc_results.append({
        'fraction': frac,
        'n_samples': n_samples,
        'train_r2': train_r2,
        'val_r2': val_r2
    })
    print(f"Fraction {frac:.1f}: train R²={train_r2:.4f}, val R²={val_r2:.4f}")

plot_learning_curve(lc_results)
print("Learning curve plot saved")

## Phase 9: Noise Experiment

In [ ]:
print("Running noise feature experiment...")
noise_counts = [0, 50, 200, 500]
noise_results = []

for n_noise in noise_counts:
    if n_noise == 0:
        Xtr_noise = X_train
        Xv_noise = X_val
    else:
        Xtr_noise = add_irrelevant_features(X_train, n_noise, seed=42)
        Xv_noise = add_irrelevant_features(X_val, n_noise, seed=42)
    
    pipeline = Pipeline([
        ('scaler', make_preprocessing_pipeline().steps[0][1]),
        ('model', ridge_regression())
    ])
    pipeline.fit(Xtr_noise, y_train)
    pred = pipeline.predict(Xv_noise)
    metrics = regression_metrics(y_val, pred)
    
    noise_results.append({'n_noise': n_noise, 'r2': metrics['r2']})
    print(f"Noise features {n_noise}: R² = {metrics['r2']:.4f}")

print("Noise experiment complete")

## Phase 10: Redundancy Experiment

In [ ]:
print("Running redundancy experiment...")

# baseline
pipeline = Pipeline([
    ('scaler', make_preprocessing_pipeline().steps[0][1]),
    ('model', ridge_regression())
])
pipeline.fit(X_train, y_train)
pred_baseline = pipeline.predict(X_val)
r2_baseline = regression_metrics(y_val, pred_baseline)['r2']
print(f"Baseline R²: {r2_baseline:.4f}")

# with redundant features
Xtr_red = add_redundant_features(X_train, n_dup=10, n_corr=10, n_perturb=10, seed=42)
Xv_red = add_redundant_features(X_val, n_dup=10, n_corr=10, n_perturb=10, seed=42)

pipeline.fit(Xtr_red, y_train)
pred_red = pipeline.predict(Xv_red)
r2_red = regression_metrics(y_val, pred_red)['r2']
print(f"With redundancy R²: {r2_red:.4f}")
print(f"Difference: {r2_red - r2_baseline:.4f}")

## Phase 11: Target Permutation Test

In [ ]:
print("Running target permutation test...")

# baseline with real target
pipeline = Pipeline([
    ('scaler', make_preprocessing_pipeline().steps[0][1]),
    ('model', ridge_regression())
])
pipeline.fit(X_train, y_train)
pred_real = pipeline.predict(X_val)
r2_real = regression_metrics(y_val, pred_real)['r2']

# with shuffled target
rng = np.random.default_rng(42)
y_train_shuffled = y_train.sample(frac=1, random_state=42).reset_index(drop=True)

pipeline.fit(X_train, y_train_shuffled)
pred_shuffled = pipeline.predict(X_val)
r2_shuffled = regression_metrics(y_val, pred_shuffled)['r2']

print(f"Real target R²: {r2_real:.4f}")
print(f"Shuffled target R²: {r2_shuffled:.4f}")
print(f"Difference: {r2_real - r2_shuffled:.4f}")
print("\nExpected: shuffled R² should be much worse (confirms no leakage)")

## Save Results Summary

In [ ]:
summary = {
    'dataset': 'atp1d',
    'n_samples': X.shape[0],
    'n_features': X.shape[1],
    'p_over_n': X.shape[1] / X.shape[0],
    'repeated_split_results': {
        model: {
            'mean_r2': float(np.mean(scores)),
            'std_r2': float(np.std(scores)),
            'min_r2': float(np.min(scores)),
            'max_r2': float(np.max(scores))
        }
        for model, scores in repeated_results.items()
    },
    'nested_cv_ridge': nested_results,
    'learning_curve': lc_results,
    'noise_experiment': noise_results,
}

logger.save_summary(summary)
print("\nResults summary saved to results/summary.json")

## Final Results Table

In [ ]:
print("\n=== FINAL MODEL COMPARISON ===")
print("\nBased on 10 repeated train/val splits:\n")

final_df = pd.DataFrame([
    {
        'Model': model,
        'Mean R²': f"{summary['repeated_split_results'][model]['mean_r2']:.4f}",
        'Std R²': f"{summary['repeated_split_results'][model]['std_r2']:.4f}",
        'Worst R²': f"{summary['repeated_split_results'][model]['min_r2']:.4f}",
    }
    for model in repeated_results.keys()
])

print(final_df.to_string(index=False))

# save to CSV
final_df.to_csv('../results/final_results.csv', index=False)
print("\nFinal results saved to results/final_results.csv")

## Conclusion

All experiments completed successfully. Results show:
1. Regularized models handle p > n effectively
2. More training data improves validation performance
3. Irrelevant features degrade performance
4. Redundant features have minimal impact on Ridge
5. Target permutation confirms leakage-free pipeline

See `report/final_report.md` for detailed analysis.